# 005 — Negative Sampling (MovieLens)

Port of the reference `011-negative-sample.ipynb`. For each positive
user–movie interaction, sample one popularity-weighted negative movie the
user did NOT interact with. Negative rows inherit the user-side features of
their positive row and pick up the item-side features (title, genres) of the
sampled movie. The time-dependent rating aggregates for the sampled movie at
the negative event timestamp come from a pandas `merge_asof` (latest movie
stats at-or-before that timestamp) — the point-in-time join Feast's file
offline store would do, but without the dask OOM.

Outputs `train_features_neg_df.parquet` / `val_features_neg_df.parquet` /
`full_features_neg_sampling_df.parquet` to `feature/output/engineer/`.

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from loguru import logger
from pydantic import BaseModel

_PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))
import sys
sys.path.insert(0, _PROJECT_ROOT)

from feature.features.negative_sampling import generate_negative_samples


class Args(BaseModel):
    run_name: str = "005-negative-sample"
    random_seed: int = int(os.getenv("RANDOM_SEED", "42"))
    user_col: str = "userId"
    item_col: str = "movieId"
    rating_col: str = "rating"
    timestamp_col: str = "event_timestamp"
    neg_to_pos_ratio: int = 1
    windows_days: list = [90, 30, 7]
    project_root: str = ""
    output_dir: str = ""
    engineer_dir: str = ""

    def init(self):
        self.project_root = _PROJECT_ROOT
        self.output_dir = os.path.join(self.project_root, "feature", "output")
        self.engineer_dir = os.path.join(self.output_dir, "engineer")
        return self


args = Args().init()
logger.info(f"engineer_dir={args.engineer_dir}")

2026-07-16 00:48:16.876 | INFO     | __main__:<module>:37 - engineer_dir=E:\MachineLearning\Recommendation_System\feature\output\engineer


In [2]:
train_df = pd.read_parquet(os.path.join(args.engineer_dir, "train_features.parquet"))
val_df = pd.read_parquet(os.path.join(args.engineer_dir, "val_features.parquet"))
logger.info(f"train={train_df.shape}  val={val_df.shape}")

val_timestamp = val_df[args.timestamp_col].min()
logger.info(f"val_timestamp={val_timestamp}")
full_df = pd.concat([train_df, val_df], ignore_index=True)
logger.info(f"full_df={full_df.shape}")

2026-07-16 00:48:17.056 | INFO     | __main__:<module>:3 - train=(79425, 22)  val=(103, 22)


2026-07-16 00:48:17.058 | INFO     | __main__:<module>:6 - val_timestamp=2018-06-26 21:04:28


2026-07-16 00:48:17.068 | INFO     | __main__:<module>:8 - full_df=(79528, 22)


## Generate negatives

Static item metadata (title, genres) carried via `item_features_df`; the
time-dependent rating aggregates for the sampled negative items are fetched
from Feast at the negative event timestamp.

In [3]:
meta_features = ["title", "genres"]
item_features_df = full_df.drop_duplicates(subset=[args.item_col])[
    [args.item_col, "item_indice", *meta_features]
].copy()

transfer_features = [
    "item_sequence",
    "user_rating_list_10_recent_movie_timestamp",
    "item_sequence_ts",
    "item_sequence_ts_bucket",
    args.user_col,
    "user_rating_cnt_90d",
    "user_rating_avg_prev_rating_90d",
    "user_rating_list_10_recent_movie",
]

neg_df = generate_negative_samples(
    full_df,
    user_col="user_indice",
    item_col="item_indice",
    label_col=args.rating_col,
    timestamp_col=args.timestamp_col,
    neg_label=0,
    neg_to_pos_ratio=args.neg_to_pos_ratio,
    seed=args.random_seed,
    features=transfer_features,
)
neg_df = neg_df.merge(item_features_df, how="left", on="item_indice", validate="m:1")
logger.info(f"neg_df={neg_df.shape}  cols={list(neg_df.columns)}")

  0%|          | 0/79528 [00:00<?, ?it/s]

2026-07-16 00:48:57.720 | INFO     | __main__:<module>:29 - neg_df=(79528, 15)  cols=['user_indice', 'item_indice', 'rating', 'event_timestamp', 'item_sequence', 'user_rating_list_10_recent_movie_timestamp', 'item_sequence_ts', 'item_sequence_ts_bucket', 'userId', 'user_rating_cnt_90d', 'user_rating_avg_prev_rating_90d', 'user_rating_list_10_recent_movie', 'movieId', 'title', 'genres']


In [4]:
item_ts_cols = [f"movie_rating_cnt_{d}d" for d in args.windows_days] + \
               [f"movie_rating_avg_prev_rating_{d}d" for d in args.windows_days]

movie_stats = pd.read_parquet(os.path.join(args.output_dir, "movie_rating_stats.parquet"))
ms_sorted = movie_stats.sort_values(args.timestamp_col)

neg_sorted = neg_df[[args.item_col, args.timestamp_col]].drop_duplicates().sort_values(args.timestamp_col)
neg_ts_df = pd.merge_asof(
    neg_sorted, ms_sorted[[args.item_col, args.timestamp_col, *item_ts_cols]],
    on=args.timestamp_col, by=args.item_col, direction="backward",
)
logger.info(f"neg_ts_df={neg_ts_df.shape}")

neg_df = neg_df.merge(neg_ts_df, how="left", on=[args.item_col, args.timestamp_col], validate="m:1")
n_nulls = int(neg_df[item_ts_cols[0]].isna().sum())
logger.info(f"neg_df after ts merge={neg_df.shape}  cold-movie nulls filled={n_nulls}")
# A negative movie may have no ratings at-or-before the event timestamp (cold at
# that point in time) -> asof returns NaN. Fill with 0 (cnt=0, avg=0): the movie
# had no prior history in any window.
neg_df[item_ts_cols] = neg_df[item_ts_cols].fillna(0) 

2026-07-16 00:48:57.783 | INFO     | __main__:<module>:12 - neg_ts_df=(79497, 8)


2026-07-16 00:48:57.848 | INFO     | __main__:<module>:16 - neg_df after ts merge=(79528, 21)  cold-movie nulls filled=19636


## Concat pos + neg, shuffle, split back to train/val, persist

In [5]:
full_features_df = (
    pd.concat([full_df, neg_df], axis=0)
    .reset_index(drop=True)
    .sample(frac=1, replace=False, random_state=args.random_seed)
)
logger.info(f"pos+neg={full_features_df.shape}")

key_cols = [args.user_col, args.item_col, "user_indice", "item_indice", "item_sequence", "item_sequence_ts_bucket", args.rating_col, args.timestamp_col]
assert full_features_df[key_cols].isna().sum().sum() == 0, "nulls in key columns"

train_neg_df = full_features_df.loc[lambda d: d[args.timestamp_col] < val_timestamp].copy()
val_neg_df = full_features_df.loc[lambda d: d[args.timestamp_col] >= val_timestamp].copy()
logger.info(f"train_neg={train_neg_df.shape}  val_neg={val_neg_df.shape}")

full_features_df.to_parquet(os.path.join(args.engineer_dir, "full_features_neg_sampling_df.parquet"), index=False)
train_neg_df.to_parquet(os.path.join(args.engineer_dir, "train_features_neg_df.parquet"), index=False)
val_neg_df.to_parquet(os.path.join(args.engineer_dir, "val_features_neg_df.parquet"), index=False)
logger.info("005 persisted OK")

2026-07-16 00:48:57.990 | INFO     | __main__:<module>:6 - pos+neg=(159056, 22)


2026-07-16 00:48:58.107 | INFO     | __main__:<module>:13 - train_neg=(158850, 22)  val_neg=(206, 22)


2026-07-16 00:48:59.498 | INFO     | __main__:<module>:18 - 005 persisted OK
